In [0]:
%run ./00_config_and_helpers

Config ready. Batch: 20261025155637_555e2c | Path: /Volumes/workspace/rag/raw


In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import date

dbutils.widgets.dropdown("gold_mode", "incremental", ["incremental", "full"])
GOLD_MODE = dbutils.widgets.get("gold_mode")

def write_overwrite(df, table):
    start = datetime.now()
    try:
        df = df.withColumn("load_timestamp", F.current_timestamp())
        n = df.count()
        df.write.format("delta").mode("overwrite").saveAsTable(table)
        log_run("Silver-to-Gold", f"{table} (overwrite)", start, "Success", inserted=n)
        print(f"{table}: {n} rows written")
    except Exception as e:
        log_run("Silver-to-Gold", f"{table} (overwrite)", start, "Failure", error=str(e)[:500])
        raise

def merge_gold(df, table, keys):
    start = datetime.now()
    try:
        df = df.withColumn("load_timestamp", F.current_timestamp())
        n_in = df.count()
        if n_in == 0:
            log_run("Silver-to-Gold", f"{table} (merge, nothing new)", start, "Success", 0, 0)
            print(f"{table}: nothing new")
            return
        if not spark.catalog.tableExists(table):
            df.write.format("delta").saveAsTable(table)
            ins, upd = n_in, 0
        else:
            tgt = DeltaTable.forName(spark, table)
            cond = " AND ".join(f"t.{k} <=> s.{k}" for k in keys)
            (tgt.alias("t").merge(df.alias("s"), cond)
                .whenMatchedUpdateAll(condition="t._silver_load_ts <> s._silver_load_ts")
                .whenNotMatchedInsertAll().execute())
            m = tgt.history(1).collect()[0]["operationMetrics"]
            ins = int(m.get("numTargetRowsInserted", 0))
            upd = int(m.get("numTargetRowsUpdated", 0))
        log_run("Silver-to-Gold", f"{table} (merge, mode={GOLD_MODE}, input={n_in})", start, "Success", ins, upd)
        print(f"{table}: {ins} inserted, {upd} updated (input {n_in})")
    except Exception as e:
        log_run("Silver-to-Gold", f"{table} (merge)", start, "Failure", error=str(e)[:500])
        raise

def watermark(table):
    if GOLD_MODE == "full" or not spark.catalog.tableExists(table):
        return None
    return spark.table(table).agg(F.max("_silver_load_ts")).first()[0]

dk = lambda c: F.date_format(c, "yyyyMMdd").cast("int")

# ---------------- dim_repo ----------------
repos = [(1, "langchain", "langchain-ai", "langchain-ai/langchain", "orchestration"),
         (2, "llama_index", "run-llama", "run-llama/llama_index", "indexing"),
         (3, "chroma", "chroma-core", "chroma-core/chroma", "vector-db"),
         (4, "qdrant", "qdrant", "qdrant/qdrant", "vector-db"),
         (5, "haystack", "deepset-ai", "deepset-ai/haystack", "framework")]
write_overwrite(spark.createDataFrame(repos, "repo_key INT, repo_name STRING, owner STRING, full_name STRING, stack_layer STRING"), "dim_repo")
dim_repo = spark.table("dim_repo").select("repo_key", "repo_name")

# ---------------- dim_date ----------------
b = spark.sql("""SELECT MIN(d) lo, MAX(d) hi FROM (
    SELECT event_date d FROM silver_commits
    UNION ALL SELECT event_date FROM silver_issues
    UNION ALL SELECT to_date(closed_ts) FROM silver_issues WHERE closed_ts IS NOT NULL)""").first()
lo, hi = b["lo"], b["hi"]
dates = spark.sql(f"SELECT explode(sequence(to_date('{lo}'), to_date('{hi}'), interval 1 day)) AS full_date")
dim_date = dates.select(
    dk("full_date").alias("date_key"), "full_date",
    F.dayofweek("full_date").alias("day_of_week"),
    F.date_format("full_date", "EEEE").alias("day_name"),
    F.weekofyear("full_date").alias("week_of_year"),
    F.month("full_date").alias("month_no"),
    F.date_format("full_date", "MMMM").alias("month_name"),
    F.date_format("full_date", "yyyy-MM").alias("year_month"),
    F.quarter("full_date").alias("quarter"),
    F.year("full_date").alias("year"),
    F.dayofweek("full_date").isin(1, 7).alias("is_weekend"))
write_overwrite(dim_date, "dim_date")

# ---------------- dim_contributor (no login, no name) ----------------
contrib_c = F.coalesce(F.sha2(F.lower(F.col("author_login")), 256), F.col("contributor_hash"), F.lit("unknown"))
c = spark.table("silver_commits").select("repo_name", "event_date",
        contrib_c.alias("contributor_hash"), F.lit(1).alias("is_commit"), F.lit(0).alias("is_issue"))
i = spark.table("silver_issues").select("repo_name", "event_date",
        F.coalesce("author_hash", F.lit("unknown")).alias("contributor_hash"),
        F.lit(0).alias("is_commit"), F.lit(1).alias("is_issue"))
dim_contributor = (c.unionByName(i).groupBy("contributor_hash").agg(
        F.min("event_date").alias("first_contribution_date"),
        F.sum("is_commit").alias("total_commits"),
        F.sum("is_issue").alias("total_issues_and_prs_opened"),
        F.countDistinct("repo_name").alias("repos_contributed_to"))
    .withColumn("is_cross_repo", F.col("repos_contributed_to") > 1))
write_overwrite(dim_contributor, "dim_contributor")

# ---------------- fact_commit_activity ----------------
s = spark.table("silver_commits")
wm = watermark("fact_commit_activity")
if wm: s = s.where(F.col("load_timestamp") > F.lit(wm))
fc = (s.join(F.broadcast(dim_repo), "repo_name", "left").select(
        "repo_key", "sha",
        dk("event_date").alias("date_key"),
        contrib_c.alias("contributor_hash"),
        F.lit(1).alias("commit_count"),
        (F.col("parent_count") > 1).alias("is_merge_commit"),
        F.coalesce("is_verified", F.lit(False)).alias("is_verified"),
        "comment_count",
        F.length("commit_message").alias("message_length"),
        "author_ts",
        F.col("load_timestamp").alias("_silver_load_ts")))
merge_gold(fc, "fact_commit_activity", ["repo_key", "sha"])

# ---------------- fact_issue_lifecycle ----------------
s = spark.table("silver_issues")
wm = watermark("fact_issue_lifecycle")
if wm: s = s.where(F.col("load_timestamp") > F.lit(wm))
fi = (s.join(F.broadcast(dim_repo), "repo_name", "left").select(
        "repo_key", "issue_id", "number",
        dk("created_ts").alias("created_date_key"),
        dk("closed_ts").alias("closed_date_key"),
        F.coalesce("author_hash", F.lit("unknown")).alias("contributor_hash"),
        "is_pull_request", "is_merged",
        (F.col("state") == "closed").alias("is_closed"),
        F.when(F.col("closed_ts").isNotNull(),
               (F.unix_timestamp("closed_ts") - F.unix_timestamp("created_ts")) / 3600.0
              ).cast("double").alias("time_to_close_hrs"),
        "comment_count", "state", "created_ts", "closed_ts",
        F.col("load_timestamp").alias("_silver_load_ts")))
merge_gold(fi, "fact_issue_lifecycle", ["repo_key", "issue_id"])

# ---------------- agg_daily_repo_health ----------------
fcA = spark.table("fact_commit_activity")
fiA = spark.table("fact_issue_lifecycle")
a_c = (fcA.where("date_key IS NOT NULL").groupBy("repo_key", "date_key")
          .agg(F.count("*").alias("commits"), F.countDistinct("contributor_hash").alias("active_contributors")))
a_o = (fiA.where("created_date_key IS NOT NULL").groupBy("repo_key", F.col("created_date_key").alias("date_key"))
          .agg(F.sum(F.when(~F.col("is_pull_request"), 1).otherwise(0)).alias("issues_opened"),
               F.sum(F.when(F.col("is_pull_request"), 1).otherwise(0)).alias("prs_opened")))
a_x = (fiA.where("closed_date_key IS NOT NULL").groupBy("repo_key", F.col("closed_date_key").alias("date_key"))
          .agg(F.count("*").alias("items_closed")))
agg = a_c.join(a_o, ["repo_key", "date_key"], "full").join(a_x, ["repo_key", "date_key"], "full").fillna(0)
write_overwrite(agg, "agg_daily_repo_health")

dim_repo: 5 rows written
dim_date: 2464 rows written
dim_contributor: 9442 rows written
fact_commit_activity: 0 inserted, 0 updated (input 17459)
fact_issue_lifecycle: 0 inserted, 0 updated (input 36927)


In [0]:
for t, cols in [("fact_commit_activity", "repo_key, date_key"),
                ("fact_issue_lifecycle", "repo_key, created_date_key"),
                ("agg_daily_repo_health", "repo_key, date_key")]:
    start = datetime.now()
    try:
        spark.sql(f"ALTER TABLE {t} CLUSTER BY ({cols})")
        spark.sql(f"OPTIMIZE {t}")
        log_run("Gold-Optimize", f"{t} CLUSTER BY ({cols})", start, "Success")
        print("optimized", t)
    except Exception as e:
        log_run("Gold-Optimize", t, start, "Failure", error=str(e)[:300])
        print("could not optimize", t, str(e)[:150])

optimized fact_commit_activity
optimized fact_issue_lifecycle
optimized agg_daily_repo_health


In [0]:
for t in ["dim_repo","dim_date","dim_contributor","fact_commit_activity","fact_issue_lifecycle","agg_daily_repo_health"]:
    print(t, spark.table(t).count())
display(spark.sql("SELECT layer, parameter_or_file, status, rows_inserted, rows_updated, start_time, end_time FROM pipeline_execution_logs WHERE layer LIKE '%Gold%' ORDER BY start_time DESC LIMIT 15"))

dim_repo 5
dim_date 2464
dim_contributor 9442
fact_commit_activity 17459
fact_issue_lifecycle 36927
agg_daily_repo_health 4943


layer,parameter_or_file,status,rows_inserted,rows_updated,start_time,end_time
Gold-Optimize,"agg_daily_repo_health CLUSTER BY (repo_key, date_key)",Success,0,0,2026-10-25T07:57:05.888Z,2026-10-25T07:57:14.581Z
Gold-Optimize,"fact_issue_lifecycle CLUSTER BY (repo_key, created_date_key)",Success,0,0,2026-10-25T07:56:58.811Z,2026-10-25T07:57:03.842Z
Gold-Optimize,"fact_commit_activity CLUSTER BY (repo_key, date_key)",Success,0,0,2026-10-25T07:56:52.185Z,2026-10-25T07:56:57.506Z
Silver-to-Gold,agg_daily_repo_health (overwrite),Success,4943,0,2026-10-25T07:56:37.490Z,2026-10-25T07:56:43.366Z
Silver-to-Gold,"fact_issue_lifecycle (merge, mode=full, input=36927)",Success,0,0,2026-10-25T07:56:28.999Z,2026-10-25T07:56:36.079Z
Silver-to-Gold,"fact_commit_activity (merge, mode=full, input=17459)",Success,0,0,2026-10-25T07:56:20.373Z,2026-10-25T07:56:27.705Z
Silver-to-Gold,dim_contributor (overwrite),Success,9442,0,2026-10-25T07:56:15.336Z,2026-10-25T07:56:18.870Z
Silver-to-Gold,dim_date (overwrite),Success,2464,0,2026-10-25T07:56:11.762Z,2026-10-25T07:56:13.910Z
Silver-to-Gold,dim_repo (overwrite),Success,5,0,2026-10-25T07:56:06.654Z,2026-10-25T07:56:08.819Z
Gold-Optimize,"agg_daily_repo_health CLUSTER BY (repo_key, date_key)",Success,0,0,2026-10-25T07:51:47.641Z,2026-10-25T07:51:56.149Z
